# From Hackathon AI to Production AI

## Chapter 03 — Can We Trust It?

Version 2.0

## Story So Far


Wednesday morning.

HelpDeskAI has now been running successfully for several days.

The engineering dashboards look excellent.

No application crashes.

No timeout issues.

No retry storms.

Everyone believes the pilot deployment is a success.

Then the customer support manager reviews yesterday's
ticket classifications.

A refund request was classified as Technical.

A login issue was classified as Billing.

The application behaved exactly as designed.

It simply produced incorrect answers.

The discussion changes immediately.

The engineering team is no longer asking whether the
application works.

They are asking whether its answers can be trusted.

## Workshop Context


In Chapter 02 we built a resilient AI application.

Our software can now recover from temporary failures,
validate input, and continue serving users.

However, reliable software can still generate poor
predictions.

Today's focus is not system reliability.

Today's focus is prediction quality.

We begin treating AI outputs as something that must be
measured rather than assumed.

## Learning Outcomes

- Create an evaluation dataset
- Compare predictions with expected labels
- Measure classification accuracy
- Identify incorrect predictions
- Understand why evaluation matters
- Think like an ML engineer

## Today's Engineering Question



How do we know whether our AI system is producing
correct and trustworthy answers?

## Engineering Principle



Reliable output

is not necessarily

correct output.

## Looking Back


Our HelpDeskAI application has evolved considerably.

Chapter 00

The model answered questions.

Chapter 01

We made the application observable.

Chapter 02

We made the application resilient.

Today we examine the quality of its decisions.

An AI application that is fast and reliable can still
produce incorrect answers.

Production AI requires both reliability and correctness.

## Engineering Discussion


Traditional software usually produces deterministic results.

Given the same input, it returns the same output.

Large Language Models behave differently.

Their outputs are generated rather than retrieved.

Small prompt changes can produce different responses.

Correct answers cannot be assumed.

They must be evaluated.

## Today's Lab


Throughout this notebook we will build a simple evaluation
pipeline.

Instead of testing one ticket at a time, we will compare
model predictions against a labelled dataset.

This mirrors how production AI teams evaluate new prompts,
models, and system improvements.

## Hands-on Exercise 1

### Import Required Libraries


Today's experiments require only a few standard Python
libraries.

In [ ]:
from google import genai
from collections import Counter

## Hands-on Exercise 2

### Configure Gemini Client


Reuse your Gemini API key from the previous chapters.

In [ ]:
try:

    from google.colab import userdata

    API_KEY = userdata.get("GOOGLE_API_KEY")

except Exception:

    API_KEY = input(
        "Enter your Gemini API Key: "
    ).strip()

client = genai.Client(
    api_key=API_KEY
)

print("Client ready.")

## Reflection


Many developers evaluate an AI application by asking
only one or two questions.

If the answers look reasonable, they assume the system is
ready.

Professional AI engineering is different.

Quality is measured across many examples rather than judged
from isolated successes.

## Hands-on Exercise 3

### Create an Evaluation Dataset


Build a small labelled dataset.

Each ticket includes the expected category determined by a
human reviewer.

This dataset will serve as our benchmark.

In [ ]:
evaluation_data = [

    {
        "ticket": "I was charged twice for my subscription.",
        "expected": "Billing"
    },

    {
        "ticket": "The application crashes during startup.",
        "expected": "Technical"
    },

    {
        "ticket": "Please change my registered email address.",
        "expected": "Account"
    },

    {
        "ticket": "I forgot my password and cannot log in.",
        "expected": "Account"
    },

    {
        "ticket": "My refund has not been processed.",
        "expected": "Billing"
    }

]

print(f"Evaluation samples: {len(evaluation_data)}")

## Engineering Discussion


Notice that every ticket has an expected answer.

These labels are often called the ground truth.

Without labelled examples there is no objective way to
measure whether an AI system is improving.

Evaluation datasets become increasingly valuable as an AI
application evolves.

## Hands-on Exercise 4

### Build a Classification Function


Create a reusable function that predicts the category for
each support ticket.

In [ ]:
def classify_ticket(ticket):

    prompt = f'''
You are a customer support classifier.

Return only one category.

Billing
Technical
Account

Ticket

{ticket}
'''

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text.strip()

## Reflection


Creating an evaluation dataset is only the first step.

The real value comes from comparing model predictions with
human-labelled answers.

Without comparison there is no measurement.

Without measurement there is no improvement.

## Hands-on Exercise 5

### Run Batch Predictions


Evaluate every ticket in the dataset.

Record both the expected label and the model prediction.

In [ ]:
results = []

for sample in evaluation_data:

    prediction = classify_ticket(sample["ticket"])

    results.append({

        "ticket": sample["ticket"],

        "expected": sample["expected"],

        "predicted": prediction

    })

print(f"Evaluated {len(results)} tickets.")

## Hands-on Exercise 6

### Review the Predictions


Inspect the expected and predicted categories.

In [ ]:
for result in results:

    print("-" * 60)

    print("Ticket")
    print(result["ticket"])

    print()

    print("Expected :", result["expected"])
    print("Predicted:", result["predicted"])

## Engineering Discussion


A prediction by itself tells us very little.

The important question is whether it matches the expected
answer.

Evaluation transforms model output into measurable evidence.

## Hands-on Exercise 7

### Calculate Classification Accuracy


Compute the percentage of correct predictions.

In [ ]:
correct = 0

for result in results:

    if result["expected"] == result["predicted"]:

        correct += 1

accuracy = correct / len(results)

print(f"Correct Predictions : {correct}")
print(f"Total Samples       : {len(results)}")
print(f"Accuracy            : {accuracy:.1%}")

## Reflection


Accuracy provides a useful summary, but it does not explain
why mistakes occur.

Professional AI engineers investigate individual errors
instead of relying only on a single percentage.

## Hands-on Exercise 8

### Inspect Incorrect Predictions


Identify every prediction that differs from the expected
label.

In [ ]:
mistakes = []

for result in results:

    if result["expected"] != result["predicted"]:

        mistakes.append(result)

if mistakes:

    for mistake in mistakes:

        print("-" * 60)

        print("Ticket")
        print(mistake["ticket"])

        print()

        print("Expected :", mistake["expected"])
        print("Predicted:", mistake["predicted"])

else:

    print("No incorrect predictions found.")

## Engineering Discussion


One incorrect prediction is often more valuable than twenty
correct ones.

Every mistake is an opportunity to improve prompts,
evaluation datasets, or system design.

Engineering teams spend much of their time studying failures
rather than celebrating successes.

## Hands-on Exercise 9

### Summarize Prediction Distribution


Look for patterns in the predicted categories.

Does the model heavily favor one category?

In [ ]:
distribution = Counter(

    result["predicted"]

    for result in results

)

print("Prediction Distribution")

for label, count in distribution.items():

    print(f"{label:12} {count}")

## Reflection


A model that predicts the same category for almost every
input might still achieve reasonable accuracy on a small
dataset.

Looking at the distribution of predictions helps reveal
biases that accuracy alone cannot detect.

## Hands-on Exercise 10

### Ground the Model with Company Policy


Provide explicit classification rules instead of relying
entirely on the model's general knowledge.

In [ ]:
def grounded_classify(ticket):

    prompt = f'''
You are HelpDeskAI.

Use ONLY the following classification policy.

Billing

- invoices
- subscriptions
- refunds
- payments

Technical

- crashes
- bugs
- errors
- performance

Account

- login
- password
- email
- profile

Return only one category.

Ticket

{ticket}
'''

    response = client.models.generate_content(

        model="gemini-2.5-flash",

        contents=prompt,

    )

    return response.text.strip()

## Hands-on Exercise 11

### Compare the Grounded Prompt


Run the evaluation again using the grounded prompt.

Did the predictions become more consistent?

Did any incorrect predictions improve?

In [ ]:
grounded_results = []

for sample in evaluation_data:

    grounded_results.append({

        "expected": sample["expected"],

        "predicted": grounded_classify(
            sample["ticket"]
        )

    })

correct = 0

for result in grounded_results:

    if result["expected"] == result["predicted"]:

        correct += 1

print(
    f"Grounded Accuracy: "
    f"{correct / len(grounded_results):.1%}"
)

## Think Like an Engineer


Prompt engineering should not be based on intuition.

Every prompt modification should be evaluated against a
representative dataset.

Improvement is demonstrated through measurement, not by
finding a single example where the new prompt performs
better.

## Reflection


Evaluation is not a one-time activity performed before
deployment.

It is a continuous engineering process.

As prompts change, models evolve, and customer behaviour
shifts, evaluation datasets should also grow and improve.

Trust must be earned continuously.

## Hands-on Exercise 12

### Compare Baseline and Grounded Results


Compare the original classifier with the grounded version.

Did the additional business rules improve the evaluation
results?

Remember that improvements should be measured, not assumed.

In [ ]:
baseline_correct = sum(
    1
    for result in results
    if result["expected"] == result["predicted"]
)

grounded_correct = sum(
    1
    for result in grounded_results
    if result["expected"] == result["predicted"]
)

print(f"Baseline Accuracy : {baseline_correct / len(results):.1%}")
print(f"Grounded Accuracy : {grounded_correct / len(grounded_results):.1%}")

difference = grounded_correct - baseline_correct

if difference > 0:
    print(f"Improvement: +{difference} correct prediction(s)")
elif difference < 0:
    print(f"Decrease: {-difference} fewer correct prediction(s)")
else:
    print("No measurable difference on this dataset.")

## Hands-on Exercise 13

### Expand the Evaluation Dataset


A five-sample dataset is useful for learning, but not for
making production decisions.

Add additional customer tickets that reflect real support
requests from your domain.

Aim for a balanced dataset that represents all categories.

In [ ]:
# Suggested activity:
#
# Add at least 10 more labelled examples to
# evaluation_data.
#
# Then rerun the evaluation pipeline and observe
# whether the accuracy changes.

## Engineering Discussion


Evaluation datasets should evolve alongside the application.

Whenever users discover incorrect predictions, consider
adding those examples to the evaluation dataset.

Over time, your benchmark becomes a valuable engineering
asset that helps prevent regressions when prompts, models,
or business rules change.

## Hands-on Exercise 14

### Analyze Misclassification Patterns


Review the mistakes identified earlier.

For each incorrect prediction, ask yourself:

• Was the ticket ambiguous?

• Was the prompt too general?

• Does the model need more domain context?

• Is the expected label itself reasonable?

Not every disagreement indicates a model failure.

In [ ]:
for mistake in mistakes:

    print("=" * 70)
    print("Ticket:")
    print(mistake["ticket"])
    print()
    print(f"Expected : {mistake['expected']}")
    print(f"Predicted: {mistake['predicted']}")
    print()
    print("Discussion:")
    print("Why might this prediction have occurred?")

## Challenge Exercise


Improve the HelpDeskAI classifier by experimenting with:

• clearer category definitions

• additional business rules

• better prompt structure

• few-shot examples

After each modification, rerun the evaluation pipeline and
record the results.

Do not rely on a single successful example.

Use the evaluation dataset to determine whether your changes
produce consistent improvements.

## Think Like an Engineer


Professional AI engineering is driven by evidence.

Instead of asking,

'I think this prompt is better.'

Engineers ask,

'What does the evaluation data show?'

Every improvement should be supported by repeatable
measurements rather than personal intuition.

## Checkpoint


Your HelpDeskAI application can now

• evaluate predictions against labelled data

• measure classification accuracy

• identify incorrect predictions

• compare prompt variations

• ground prompts using business rules

• improve prompts using measurable evidence

The application is no longer just functional.

Its quality can now be evaluated systematically.

## Chapter Summary


This chapter introduced one of the defining practices of
production AI engineering: evaluation.

Reliable software is not automatically trustworthy.

By creating labelled datasets, comparing predictions with
expected answers, calculating accuracy, and analysing
mistakes, we transformed AI quality into something that can
be measured and improved.

Evaluation pipelines allow engineering teams to make
evidence-based decisions instead of relying on intuition.

Trustworthy AI is built through continuous measurement,
analysis, and refinement.

## Production Readiness Checklist

Completed

☑ Evaluation dataset
☑ Expected labels
☑ Batch evaluation
☑ Accuracy measurement
☑ Error analysis
☑ Grounded prompts
☑ Prompt comparison
☑ Quality evaluation

Still Missing

☐ Human review workflow
☐ Production monitoring
☐ Model versioning
☐ A/B testing
☐ Deployment strategy
☐ Security review
☐ Cost optimization
☐ Operational governance

## Looking Ahead

### Chapter 04 — Would We Ship It?


HelpDeskAI now answers questions.

It is observable.

It is resilient.

Its quality can be measured.

One final question remains.

Would you deploy this system for thousands of real users?

The final chapter explores deployment decisions, monitoring,
cost, scalability, governance, and the engineering trade-offs
required before an AI application is truly production-ready.

## Reflection


Chapter 00 asked

Can it answer?

Chapter 01 asked

Do we understand it?

Chapter 02 asked

Will it survive reality?

Chapter 03 asked

Can we trust it?

Production AI requires every one of these questions to be
answered before software reaches real users.